In [5]:
import pandas as pd
import numpy as np
import time

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, classification_report

import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense
from tensorflow.keras.optimizers import Adam

In [6]:
file_path = "/content/netsec.csv"   # change to your file name

# Get column names from #fields line
with open(file_path, "r") as f:
    for line in f:
        if line.startswith("#fields"):
            columns = line.strip().split("\t")[1:]
            break

df = pd.read_csv(
    file_path,
    sep="\t",
    comment="#",
    names=columns
)

df.columns = df.columns.str.strip()

print("Original dataset shape:", df.shape)
print(df.head())
print(df.columns.tolist())

Original dataset shape: (23145, 23)
             ts                 uid      id.orig_h  id.orig_p       id.resp_h  \
0  1.545404e+09   CrDn63WjJEmrWGjqf  192.168.1.195      41040  185.244.25.235   
1  1.545404e+09  CY9lJW3gh1Eje4usP6  192.168.1.195      41040  185.244.25.235   
2  1.545404e+09   CcFXLynukEDnUlvgl  192.168.1.195      41040  185.244.25.235   
3  1.545404e+09   CDrkrSobGYxHhYfth  192.168.1.195      41040  185.244.25.235   
4  1.545404e+09  CTWZQf2oJSvq6zmPAc  192.168.1.195      41042  185.244.25.235   

   id.resp_p proto service  duration orig_bytes  ... local_resp missed_bytes  \
0         80   tcp       -  3.139211          0  ...          -            0   
1         80   tcp       -         -          -  ...          -            0   
2         80   tcp       -         -          -  ...          -            0   
3         80   tcp    http  1.477656        149  ...          -         2896   
4         80   tcp       -  3.147116          0  ...          -            0 

In [7]:
# Replace Zeek missing values
df = df.replace("-", 0)

# Final balanced feature set
features = [
    "duration",
    "orig_bytes",
    "resp_bytes",
    "orig_pkts",
    "resp_pkts",
    "conn_state",
    "id.orig_p",
    "id.resp_p"
]

df_model = df[features + ["label"]].copy()

# Encode categorical feature
df_model["conn_state"] = df_model["conn_state"].astype("category").cat.codes

# Convert labels:
# Benign = 0
# Attack/Malicious = 1
df_model["label"] = df_model["label"].apply(
    lambda x: 0 if str(x).strip() == "Benign" else 1
)

# Convert features to numeric
for col in features:
    df_model[col] = pd.to_numeric(df_model[col], errors="coerce")

# Fill missing values
df_model = df_model.fillna(0)

print("Cleaned dataset shape:", df_model.shape)
print("\nLabel distribution:")
print(df_model["label"].value_counts())

print("\nPreview:")
print(df_model.head())

Cleaned dataset shape: (23145, 9)

Label distribution:
label
1    21222
0     1923
Name: count, dtype: int64

Preview:
   duration  orig_bytes  resp_bytes  orig_pkts  resp_pkts  conn_state  \
0  3.139211           0           0          3          0           2   
1  0.000000           0           0          1          0           2   
2  0.000000           0           0          1          0           2   
3  1.477656         149      128252         94         96           5   
4  3.147116           0           0          3          0           2   

   id.orig_p  id.resp_p  label  
0      41040         80      0  
1      41040         80      0  
2      41040         80      0  
3      41040         80      0  
4      41042         80      0  


/tmp/ipykernel_8655/4185653729.py:2: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df = df.replace("-", 0)


Prepare X and y

In [8]:
X = df_model.drop(columns=["label"])
y = df_model["label"]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Train Autoencoder only on benign traffic
X_train_normal = X_scaled[y == 0]

print("Total samples:", X_scaled.shape[0])
print("Normal training samples:", X_train_normal.shape[0])
print("Number of features:", X_scaled.shape[1])

Total samples: 23145
Normal training samples: 1923
Number of features: 8


Build Autoencoder Model

In [9]:
input_dim = X_train_normal.shape[1]

input_layer = Input(shape=(input_dim,))

# Encoder
encoded = Dense(16, activation="relu")(input_layer)
encoded = Dense(8, activation="relu")(encoded)

# Decoder
decoded = Dense(16, activation="relu")(encoded)
decoded = Dense(input_dim, activation="linear")(decoded)

autoencoder = Model(inputs=input_layer, outputs=decoded)

autoencoder.compile(
    optimizer=Adam(learning_rate=0.001),
    loss="mse"
)

autoencoder.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 8)              │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 16)             │           144 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 8)              │           136 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 16)             │           144 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 8)              │           136 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 560 (2.19 KB)

 Trainable params: 560 (2.19 KB)

 Non-trainable params: 0 (0.00 B)

Train Autoencoder

In [10]:
start_train = time.time()

history = autoencoder.fit(
    X_train_normal,
    X_train_normal,
    epochs=20,
    batch_size=256,
    shuffle=True,
    validation_split=0.2,
    verbose=1
)

training_time = time.time() - start_train

print("Training time:", training_time)

Epoch 1/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 2s 57ms/step - loss: 3.9299 - val_loss: 2.1331
Epoch 2/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - loss: 3.7070 - val_loss: 2.0784
Epoch 3/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - loss: 3.5961 - val_loss: 2.0300
Epoch 4/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step - loss: 3.5049 - val_loss: 1.9778
Epoch 5/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - loss: 3.4279 - val_loss: 1.9197
Epoch 6/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - loss: 3.3412 - val_loss: 1.8555
Epoch 7/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - loss: 3.2478 - val_loss: 1.7852
Epoch 8/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step - loss: 3.1412 - val_loss: 1.7074
Epoch 9/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - loss: 3.0175 - val_loss: 1.6143
Epoch 10/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - loss: 2.8710 - val_loss: 1.5139
Epoch 11/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step - loss: 2.7076 - val_loss: 1.3902
Epoch 12/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - loss: 2.5243 - val_loss: 1.2489
E

Reconstruction Error

In [11]:
start_infer = time.time()

reconstructions = autoencoder.predict(X_scaled)

inference_time = time.time() - start_infer

mse = np.mean(np.power(X_scaled - reconstructions, 2), axis=1)

print("Total inference time:", inference_time)
print("Latency per flow:", inference_time / len(X_scaled))

print("MSE preview:")
print(mse[:10])

724/724 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step
Total inference time: 1.9103891849517822
Latency per flow: 8.254003823511697e-05
MSE preview:
[7.58417002e-02 7.67319472e-02 7.67319472e-02 1.81534208e+02
 7.58453508e-02 1.82995118e+02 9.33786533e+01 1.09378172e+02
 7.58686422e-02 7.67536513e-02]


Set Threshold

In [12]:
threshold = np.percentile(mse, 90)

print("Threshold:", threshold)

Threshold: 0.23954843810310505


Predict Anomalies

In [13]:
# If reconstruction error is high, classify as anomaly
y_pred = [1 if error > threshold else 0 for error in mse]

Evaluate Autoencoder

In [14]:
cm = confusion_matrix(y, y_pred)

print("Confusion Matrix:")
print(cm)

print("\nClassification Report:")
print(classification_report(y, y_pred))

tn, fp, fn, tp = cm.ravel()

fpr = fp / (fp + tn)
recall_attack = tp / (tp + fn)
precision_attack = tp / (tp + fp)

print("\nFalse Positive Rate:", fpr)
print("Attack Recall:", recall_attack)
print("Attack Precision:", precision_attack)
print("Training time:", training_time)
print("Total inference time:", inference_time)
print("Latency per flow:", inference_time / len(X_scaled))

Confusion Matrix:
[[ 1860    63]
 [19425  1797]]

Classification Report:
              precision    recall  f1-score   support

           0       0.09      0.97      0.16      1923
           1       0.97      0.08      0.16     21222

    accuracy                           0.16     23145
   macro avg       0.53      0.53      0.16     23145
weighted avg       0.89      0.16      0.16     23145


False Positive Rate: 0.0327613104524181
Attack Recall: 0.08467627933276788
Attack Precision: 0.9661290322580646
Training time: 5.220300912857056
Total inference time: 1.9103891849517822
Latency per flow: 8.254003823511697e-05


Threshold Tuning

In [15]:
for percentile in [85, 90, 92, 95, 97]:

    threshold = np.percentile(mse, percentile)
    y_pred = [1 if error > threshold else 0 for error in mse]

    cm = confusion_matrix(y, y_pred)
    tn, fp, fn, tp = cm.ravel()

    fpr = fp / (fp + tn)
    recall_attack = tp / (tp + fn)
    precision_attack = tp / (tp + fp)

    print("\n==============================")
    print("Threshold Percentile:", percentile)
    print("Threshold:", threshold)
    print("Confusion Matrix:")
    print(cm)
    print("FPR:", fpr)
    print("Attack Recall:", recall_attack)
    print("Attack Precision:", precision_attack)


Threshold Percentile: 85
Threshold: 0.23954843810310505
Confusion Matrix:
[[ 1860    63]
 [19425  1797]]
FPR: 0.0327613104524181
Attack Recall: 0.08467627933276788
Attack Precision: 0.9661290322580646

Threshold Percentile: 90
Threshold: 0.23954843810310505
Confusion Matrix:
[[ 1860    63]
 [19425  1797]]
FPR: 0.0327613104524181
Attack Recall: 0.08467627933276788
Attack Precision: 0.9661290322580646

Threshold Percentile: 92
Threshold: 0.2729345075211989
Confusion Matrix:
[[ 1860    63]
 [19433  1789]]
FPR: 0.0327613104524181
Attack Recall: 0.084299312034681
Attack Precision: 0.9659827213822895

Threshold Percentile: 95
Threshold: 0.38556968482109866
Confusion Matrix:
[[ 1872    51]
 [20115  1107]]
FPR: 0.0265210608424337
Attack Recall: 0.05216284987277354
Attack Precision: 0.9559585492227979

Threshold Percentile: 97
Threshold: 0.39496353213576346
Confusion Matrix:
[[ 1873    50]
 [20577   645]]
FPR: 0.026001040041601663
Attack Recall: 0.030392988408255585
Attack Precision: 0.9280575